In [1]:
import torch
import time
import warnings
import os
import sys
from numba.core.errors import NumbaWarning

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from device import setup_device
from datasets.loading import get_data_loaders
from model.loading import load_model
from train.predictor import train_predictor
from analysis.visualization import visualize
from analysis.evaluation import evaluate_on_loader
from analysis.extraction import (
    extract_embeddings_from_model,
    extract_all_embeddings_from_model
)
from utils.checkpoint import save_checkpoint
from utils.config_io import save_configuration
from utils.cleanup import clear_gpu_cache, run_gc
from utils.printing import print_verbose
from utils.seed import set_seed
from setup.model_setup import setup_predictor
from setup.test_dir_setup import setup_testing_directory
from setup.training_setup import setup_training_environment, setup_train_config

In [2]:
warnings.filterwarnings("ignore", message=".*force_all_finite.*")
warnings.filterwarnings("ignore", category=NumbaWarning)
warnings.filterwarnings("ignore", message=".*verbose parameter is deprecated.*")
warnings.filterwarnings("ignore", message=".*epoch parameter in `scheduler.step.*")

In [3]:
clear_gpu_cache()
run_gc()
set_seed(0)

[03:27:19] [INFO] 🧹 Cleared GPU cache.
[03:27:19] [INFO] 🗑️  Garbage collector freed 16706 objects.
[03:27:19] [INFO] 🌱 Random seed set to 0.


In [4]:
config = {
    'verbose': True,
    'seed': seed,
    'use_gpu_1_only': False,

    'backbone_model': 'resnet18',
    'encoder_output_dim': 128,
    'pretrained': False,
    'freeze_encoder': True,
    'predictor_output_dim': 1,
    
    'base_dir': '../outputs/current',
    'test_name': 'resnet_18_regression_predictor_test',
    'best_model_file_name': 'best_model.pth',
    'trained_encoder_file': '', #TODO: add reference to trained encoder
    
    'data_folder': '../data/utkface/UTKFace',
    'augmentations': 'crop,flip,color,grayscale',
    'split': 'default',
    'train_size': 0.7,
    'val_size': 0.15,
    'test_size': 0.15,

    'batch_size': 512,
    'num_epochs': 200,
    'num_workers': 16,
    'learning_rate': 0.01, #TODO: add optimum
    'optimizer': 'sgd',
    'weight_decay': 1e-6, #TODO: add optimum
    'momentum': 0.9,
    'scheduler': 'cosine',
    'temperature': 2.0, #TODO: add optimum
    
    'key_metric': 'val_loss',
    'mode': 'min',
    'save_best_model': True,
    'use_early_stopping': True,
    'patience': 10,
    'delta': 1e-4,
    'save_intermediate_models': False,
    'write_to_tensorboard': True,
    'training_metrics': [
        'val_loss', "embedding_norm", "embedding_variance", "knn_accuracy",
        "knr_error", "spearman", "kendall", "grad_norm", "lr"
    ],
    'nearest_neighbors': 5 # for knn or knr analysis
}

In [5]:
setup_testing_directory(config, create_unique_dir=True)
save_configuration(config)

[03:27:19] [INFO] ✅ Test directory created: ../outputs/current/200_epochs_resnet18_20250529_0327
[03:27:19] [INFO] 📝 Configuration saved to ../outputs/current/200_epochs_resnet18_20250529_0327/config.json


In [6]:
device, predictor, optimizer, scheduler, monitor = setup_training_environment({**config, 'model_builder': setup_predictor})

train_loader, val_loader, train_loader_clean, test_loader = get_data_loaders(
    config['data_folder'], config['augmentations'], 
    batch_size=config['batch_size'], train_size=config['train_size']
)
criterion = torch.nn.L1Loss()

train_config = setup_train_config(
    device, predictor, optimizer, scheduler, monitor,
    train_loader, val_loader, train_loader_clean,
    config
)

[03:27:19] [INFO] ✅ CUDA is available: 2 GPU(s) visible
[03:27:19] [INFO] 🚀 Wrapping model in DataParallel over 2 GPUs
[03:27:19] [INFO] Enabled monitors: ['LoggingMonitor', 'OptimumMonitor', 'SaveModelMonitor', 'TensorboardMonitor']


In [7]:
# TRAINING CELL

start_time = time.time()
train_predictor({**train_config, 'criterion': criterion}, verbose=config['verbose'])
end_time = time.time()

print_verbose(f"Predictor training completed in: {end_time - start_time:.2f} seconds", config['verbose'])

[03:27:20] [INFO] Configuration check passed: 7 keys found.
[03:27:20] [INFO] Configuration check passed: 9 keys found.
[03:27:20] [INFO] Training on device: cuda


Extracting Embeddings: 100%|██████████████████████████████| 75/75 [ 7.88batch/s]


[03:28:26] [INFO] [Epoch 1] train_loss: 5.2740 (min)
[03:28:26] [INFO] ✅ New best train_loss: 5.2740 (min)


Extracting Embeddings: 100%|██████████████████████████████| 75/75 [ 8.19batch/s]


[03:29:33] [INFO] [Epoch 2] train_loss: 5.2676 (min)
[03:29:33] [INFO] ✅ New best train_loss: 5.2676 (min)


...

[07:12:57] [INFO] [Epoch 198] train_loss: 4.9804 (min)


Extracting Embeddings: 100%|██████████████████████████████| 75/75 [ 7.96batch/s]


[07:14:07] [INFO] [Epoch 199] train_loss: 4.9829 (min)


Extracting Embeddings: 100%|██████████████████████████████| 75/75 [ 7.99batch/s]


[07:15:16] [INFO] [Epoch 200] train_loss: 4.9826 (min)
[07:15:16] [INFO] Training complete!
[07:15:16] [INFO] Training completed in: 13676.77 seconds


In [ ]:
# ANALYSIS CELL

results = evaluate_on_loader(
    model=predictor,
    loader=test_loader,
    criterion=criterion,
    device=config['device']
)

print(f"Test Loss: {results['test_loss']:.4f}")
if "test_accuracy" in results:
    print(f"Test Accuracy: {results['test_accuracy']*100:.2f}%")